# 3D Electromagnetic Modeling of an Underwater Archaeological Site

This notebook demonstrates 3D controlled-source electromagnetic (CSEM) forward modeling for an underwater archaeological site. We use [SimPEG](https://simpeg.xyz/) and [discretize](https://discretize.simpeg.xyz/) for mesh construction and EM simulation, with [matplotlib](https://matplotlib.org/) for visualization.

**Scenario:** A metallic shipwreck or artifact cluster buried beneath the seafloor is detected using a horizontal electric dipole (HED) source and an array of electric-field receivers towed along the seafloor.

---
## Outline
1. [Install & Import Libraries](#1.-Install-&-Import-Libraries)
2. [Define the 3D Mesh](#2.-Define-the-3D-Mesh)
3. [Build the Conductivity Model](#3.-Build-the-Conductivity-Model)
4. [Set Up the CSEM Survey](#4.-Set-Up-the-CSEM-Survey)
5. [Run the Forward Simulation](#5.-Run-the-Forward-Simulation)
6. [Visualize the Results](#6.-Visualize-the-Results)
7. [Anomaly Detection](#7.-Anomaly-Detection)

## 1. Install & Import Libraries

In [ ]:
# Install required packages (run once)
# !pip install simpeg discretize numpy scipy matplotlib

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.patches import Rectangle
from mpl_toolkits.axes_grid1 import make_axes_locatable

import discretize
from discretize import TensorMesh

from SimPEG import maps
from SimPEG.electromagnetics import frequency_domain as FDEM
from SimPEG.electromagnetics.utils import conductive_sphere_in_halfspace

# Reproducibility
np.random.seed(42)

print("All libraries loaded successfully.")

## 2. Define the 3D Mesh

We build a **tensor mesh** centred on the survey area. The domain is split into three main layers:

| Layer | Description | σ (S/m) |
|-------|-------------|----------|
| Air | Above sea surface | 10⁻⁸ |
| Seawater | Water column (50 m deep) | 3.2 |
| Seafloor sediments | Marine sediments | 1.0 |
| Archaeological anomaly | Metallic shipwreck/artifacts | 10⁴ |

In [ ]:
# ------------------------------------------------------------------
# Mesh parameters
# ------------------------------------------------------------------

# Core cell sizes (m)
dh = 5.0   # horizontal
dv = 5.0   # vertical

# Number of core cells in each direction
nx_core = 40
ny_core = 40
nz_core = 30

# Padding cells (exponential growth factor)
pad_factor = 1.4
n_pad = 8

def _make_axis(n_core, d_core, n_pad, factor):
    """Build a 1-D cell-width array: padding | core | padding."""
    core = np.ones(n_core) * d_core
    pad  = d_core * factor ** np.arange(1, n_pad + 1)
    return np.r_[pad[::-1], core, pad]

hx = _make_axis(nx_core, dh, n_pad, pad_factor)
hy = _make_axis(ny_core, dh, n_pad, pad_factor)
hz = _make_axis(nz_core, dv, n_pad, pad_factor)

mesh = TensorMesh([hx, hy, hz], origin="CCN")
print(mesh)

# Convenient aliases
xC, yC, zC = mesh.cell_centers[:, 0], mesh.cell_centers[:, 1], mesh.cell_centers[:, 2]

print(f"\nMesh extents:")
print(f"  X: [{mesh.nodes_x.min():.0f}, {mesh.nodes_x.max():.0f}] m")
print(f"  Y: [{mesh.nodes_y.min():.0f}, {mesh.nodes_y.max():.0f}] m")
print(f"  Z: [{mesh.nodes_z.min():.0f}, {mesh.nodes_z.max():.0f}] m")

## 3. Build the Conductivity Model

The geological/environmental model is built layer by layer in the `z` direction and a compact anomaly is added to represent the archaeological target.

In [ ]:
# ------------------------------------------------------------------
# Conductivity values (S/m)
# ------------------------------------------------------------------
sig_air    = 1e-8   # air (effectively insulating)
sig_water  = 3.2    # seawater
sig_sed    = 1.0    # marine sediments
sig_target = 1e4    # metallic shipwreck / artifact cluster
sig_bedrock = 0.01  # consolidated rock / bedrock

# ------------------------------------------------------------------
# Layer geometry (z is positive upward, origin at sea surface = 0 m)
# ------------------------------------------------------------------
z_sea_surface  =   0.0   # sea surface
z_seafloor     = -50.0   # seafloor depth
z_sediment_bot = -80.0   # base of sediment layer

# ------------------------------------------------------------------
# Archaeological target (a box-shaped anomaly)
# ------------------------------------------------------------------
target_x = [-15, 15]   # m
target_y = [-15, 15]   # m
target_z = [-62, -55]  # m (buried 5–12 m below seafloor)

# ------------------------------------------------------------------
# Assign conductivity to every cell
# ------------------------------------------------------------------
sigma = np.ones(mesh.n_cells) * sig_air

# Seawater
sigma[(zC <= z_sea_surface) & (zC > z_seafloor)] = sig_water

# Marine sediments
sigma[(zC <= z_seafloor) & (zC > z_sediment_bot)] = sig_sed

# Bedrock
sigma[zC <= z_sediment_bot] = sig_bedrock

# Archaeological target (overrides sediment)
target_mask = (
    (xC >= target_x[0]) & (xC <= target_x[1]) &
    (yC >= target_y[0]) & (yC <= target_y[1]) &
    (zC >= target_z[0]) & (zC <= target_z[1])
)
sigma[target_mask] = sig_target

# Background model (no target) for anomaly comparison later
sigma_bg = sigma.copy()
sigma_bg[target_mask] = sig_sed

print(f"Model cells: {mesh.n_cells}")
print(f"Target cells: {target_mask.sum()}")

In [ ]:
# ------------------------------------------------------------------
# Quick cross-section plot of the conductivity model
# ------------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- X-Z slice through Y=0 ---
ax = axes[0]
ind_y = int(mesh.shape_cells[1] / 2)  # central Y slice
im = mesh.plot_slice(
    np.log10(sigma),
    ax=ax,
    normal="Y",
    ind=ind_y,
    pcolor_opts={"cmap": "RdYlBu_r", "vmin": -8, "vmax": 4},
    grid=False,
)
ax.set_xlim(-150, 150)
ax.set_ylim(-120, 20)
ax.set_xlabel("X (m)")
ax.set_ylabel("Z (m)")
ax.set_title("Conductivity model – X-Z slice (Y=0)")
ax.axhline(z_sea_surface, color="cyan", lw=1.5, linestyle="--", label="Sea surface")
ax.axhline(z_seafloor, color="brown", lw=1.5, linestyle="--", label="Seafloor")
ax.legend(loc="upper right", fontsize=8)
rect = Rectangle(
    (target_x[0], target_z[0]),
    target_x[1] - target_x[0],
    target_z[1] - target_z[0],
    linewidth=2, edgecolor="lime", facecolor="none", label="Target"
)
ax.add_patch(rect)

divider = make_axes_locatable(ax)
cax = divider.append_axes("right", size="3%", pad=0.1)
cb = plt.colorbar(im[0], cax=cax)
cb.set_label("log₁₀(σ) [S/m]")

# --- Y-Z slice through X=0 ---
ax = axes[1]
ind_x = int(mesh.shape_cells[0] / 2)  # central X slice
im = mesh.plot_slice(
    np.log10(sigma),
    ax=ax,
    normal="X",
    ind=ind_x,
    pcolor_opts={"cmap": "RdYlBu_r", "vmin": -8, "vmax": 4},
    grid=False,
)
ax.set_xlim(-150, 150)
ax.set_ylim(-120, 20)
ax.set_xlabel("Y (m)")
ax.set_ylabel("Z (m)")
ax.set_title("Conductivity model – Y-Z slice (X=0)")
ax.axhline(z_sea_surface, color="cyan", lw=1.5, linestyle="--")
ax.axhline(z_seafloor, color="brown", lw=1.5, linestyle="--")
rect2 = Rectangle(
    (target_y[0], target_z[0]),
    target_y[1] - target_y[0],
    target_z[1] - target_z[0],
    linewidth=2, edgecolor="lime", facecolor="none"
)
ax.add_patch(rect2)

divider = make_axes_locatable(ax)
cax = divider.append_axes("right", size="3%", pad=0.1)
cb = plt.colorbar(im[0], cax=cax)
cb.set_label("log₁₀(σ) [S/m]")

plt.tight_layout()
plt.savefig("conductivity_model.png", dpi=150, bbox_inches="tight")
plt.show()
print("Conductivity model plotted.")

## 4. Set Up the CSEM Survey

We use a **horizontal electric dipole (HED)** source at 5 m above the seafloor (z = −45 m) oriented in the x-direction. Receivers measure the x-component of the electric field (Ex) along a tow line on the seafloor.

Multiple frequencies are simulated to build a frequency-domain data set.

In [ ]:
# ------------------------------------------------------------------
# Survey geometry
# ------------------------------------------------------------------
frequencies = np.array([0.1, 0.5, 1.0, 5.0])  # Hz

z_source   = z_seafloor + 5.0   # 5 m above seafloor
z_receiver = z_seafloor         # on the seafloor

# Receiver line along X, centred at Y=0
rx_x = np.linspace(-150, 150, 61)  # 61 receivers, 5 m spacing
rx_y = np.zeros_like(rx_x)
rx_z = np.full_like(rx_x, z_receiver)
rx_locations = np.c_[rx_x, rx_y, rx_z]

# Source at origin (tow position above target); 1-m HED in x-direction
hed_length = 1.0  # dipole half-length (m)
hed_nodes = np.array([
    [-hed_length / 2, 0.0, z_source],
    [ hed_length / 2, 0.0, z_source],
])

# Build SimPEG survey objects
source_list = []
for freq in frequencies:
    # Electric field receiver (Ex component)
    rx = FDEM.Rx.PointElectricField(
        locations=rx_locations,
        orientation="x",
        component="real",
    )
    rx_imag = FDEM.Rx.PointElectricField(
        locations=rx_locations,
        orientation="x",
        component="imag",
    )
    # HED source – a short line current in the x-direction
    src = FDEM.Src.LineCurrent(
        receiver_list=[rx, rx_imag],
        frequency=freq,
        nodes=hed_nodes,
    )
    source_list.append(src)

survey = FDEM.Survey(source_list)
print(f"Survey created: {len(frequencies)} frequencies, {len(rx_x)} receivers each")
print(f"Source nodes: {hed_nodes}")
print(f"Receiver offsets: {rx_x.min():.0f} to {rx_x.max():.0f} m")

## 5. Run the Forward Simulation

We use the `Simulation3DMagneticFluxDensity` (finite-volume frequency-domain) solver. Two simulations are run:
- **Full model** – includes the archaeological target
- **Background model** – no target (reference)

The ratio of the two gives the **normalized anomaly**, which highlights the presence of the target.

In [ ]:
# ------------------------------------------------------------------
# Build the forward simulation
# ------------------------------------------------------------------
sigma_map = maps.IdentityMap(nP=mesh.n_cells)

sim = FDEM.Simulation3DElectricField(
    mesh=mesh,
    survey=survey,
    sigmaMap=sigma_map,
    solver="Pardiso",   # use LU solver (falls back to scipy if Pardiso absent)
)

print("Running forward simulation (full model with target) …")
dpred_full = sim.dpred(sigma)
print("  Done.")

print("Running forward simulation (background model) …")
dpred_bg = sim.dpred(sigma_bg)
print("  Done.")

# Reshape: (n_freqs * 2 * n_rx,) -> (n_freqs, 2, n_rx)
n_rx   = len(rx_x)
n_freq = len(frequencies)
dpred_full = dpred_full.reshape(n_freq, 2, n_rx)
dpred_bg   = dpred_bg.reshape(n_freq, 2, n_rx)

## 6. Visualize the Results

### 6.1 Electric field amplitude vs. receiver offset

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10), sharex=True)
axes = axes.flatten()

colors = plt.cm.viridis(np.linspace(0.1, 0.9, n_freq))

for i, (freq, color) in enumerate(zip(frequencies, colors)):
    ax = axes[i]
    # Amplitude = sqrt(Re^2 + Im^2)
    amp_full = np.sqrt(dpred_full[i, 0, :] ** 2 + dpred_full[i, 1, :] ** 2)
    amp_bg   = np.sqrt(dpred_bg[i, 0, :] ** 2 + dpred_bg[i, 1, :] ** 2)

    ax.semilogy(rx_x, amp_full, "b-",  lw=2, label="Full (with target)")
    ax.semilogy(rx_x, amp_bg,   "r--", lw=2, label="Background")
    ax.axvline(0, color="gray", lw=1, linestyle=":")
    ax.set_title(f"f = {freq} Hz")
    ax.set_ylabel("|Ex| (V/m)")
    ax.legend(fontsize=8)
    ax.grid(True, which="both", alpha=0.3)

for ax in axes[-2:]:
    ax.set_xlabel("Receiver offset X (m)")

fig.suptitle("Electric field amplitude – full model vs. background", fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig("electric_field_amplitude.png", dpi=150, bbox_inches="tight")
plt.show()

### 6.2 Normalised anomaly (full / background − 1)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10), sharex=True)
axes = axes.flatten()

for i, (freq, color) in enumerate(zip(frequencies, colors)):
    ax = axes[i]
    amp_full = np.sqrt(dpred_full[i, 0, :] ** 2 + dpred_full[i, 1, :] ** 2)
    amp_bg   = np.sqrt(dpred_bg[i, 0, :] ** 2 + dpred_bg[i, 1, :] ** 2)
    anomaly_pct = (amp_full / amp_bg - 1.0) * 100.0

    ax.plot(rx_x, anomaly_pct, color=color, lw=2)
    ax.axhline(0, color="k", lw=0.8)
    ax.axvline(0, color="gray", lw=1, linestyle=":")
    ax.fill_between(rx_x, anomaly_pct, where=(anomaly_pct > 0),
                    alpha=0.3, color="green", label="+anomaly")
    ax.fill_between(rx_x, anomaly_pct, where=(anomaly_pct < 0),
                    alpha=0.3, color="red", label="−anomaly")
    ax.set_title(f"f = {freq} Hz")
    ax.set_ylabel("Anomaly (%)")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

for ax in axes[-2:]:
    ax.set_xlabel("Receiver offset X (m)")

fig.suptitle("Normalised EM anomaly due to archaeological target", fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig("normalised_anomaly.png", dpi=150, bbox_inches="tight")
plt.show()

## 7. Anomaly Detection

### 7.1 Phase response

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10), sharex=True)
axes = axes.flatten()

for i, (freq, color) in enumerate(zip(frequencies, colors)):
    ax = axes[i]
    phase_full = np.degrees(np.arctan2(dpred_full[i, 1, :], dpred_full[i, 0, :]))
    phase_bg   = np.degrees(np.arctan2(dpred_bg[i, 1, :],   dpred_bg[i, 0, :]))

    ax.plot(rx_x, phase_full, "b-",  lw=2, label="Full (with target)")
    ax.plot(rx_x, phase_bg,   "r--", lw=2, label="Background")
    ax.axvline(0, color="gray", lw=1, linestyle=":")
    ax.set_title(f"f = {freq} Hz")
    ax.set_ylabel("Phase (°)")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

for ax in axes[-2:]:
    ax.set_xlabel("Receiver offset X (m)")

fig.suptitle("Electric field phase – full model vs. background", fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig("phase_response.png", dpi=150, bbox_inches="tight")
plt.show()

### 7.2 Multi-frequency anomaly map (receiver offset vs. frequency)

In [ ]:
# Build anomaly matrix: shape (n_freq, n_rx)
anomaly_matrix = np.zeros((n_freq, n_rx))
for i in range(n_freq):
    amp_full = np.sqrt(dpred_full[i, 0, :] ** 2 + dpred_full[i, 1, :] ** 2)
    amp_bg   = np.sqrt(dpred_bg[i, 0, :] ** 2 + dpred_bg[i, 1, :] ** 2)
    anomaly_matrix[i, :] = (amp_full / amp_bg - 1.0) * 100.0

fig, ax = plt.subplots(figsize=(12, 4))
extent = [rx_x.min(), rx_x.max(), frequencies.min(), frequencies.max()]
im = ax.imshow(
    anomaly_matrix,
    aspect="auto",
    extent=extent,
    origin="lower",
    cmap="seismic",
    vmin=-np.abs(anomaly_matrix).max(),
    vmax= np.abs(anomaly_matrix).max(),
)
ax.set_xlabel("Receiver offset X (m)")
ax.set_ylabel("Frequency (Hz)")
ax.set_title("Multi-frequency normalised EM anomaly (%)")
ax.axvline(0, color="lime", lw=1.5, linestyle="--", label="Target centre")
ax.legend(fontsize=9)
cb = plt.colorbar(im, ax=ax, pad=0.01)
cb.set_label("Anomaly (%)")
plt.tight_layout()
plt.savefig("multifreq_anomaly_map.png", dpi=150, bbox_inches="tight")
plt.show()

### 7.3 Summary table

In [ ]:
print("\n=== Anomaly Summary ===")
print(f"{'Freq (Hz)':>12}  {'Max +anomaly (%)':>18}  {'Max |anomaly| (%)':>20}")
print("-" * 55)
for i, freq in enumerate(frequencies):
    row = anomaly_matrix[i, :]
    print(f"{freq:>12.1f}  {row.max():>18.2f}  {np.abs(row).max():>20.2f}")

print("\nConclusion:")
best_i = np.argmax([np.abs(anomaly_matrix[i]).max() for i in range(n_freq)])
print(f"  Best detection frequency: {frequencies[best_i]} Hz")
print(f"  Maximum anomaly amplitude: {np.abs(anomaly_matrix[best_i]).max():.1f}%")

## Summary

This notebook performed a complete **3D frequency-domain CSEM forward modeling** workflow for an underwater archaeological site:

1. **Mesh** – A 3D tensor mesh with fine core cells (5 m) and expanding padding to absorb boundary reflections.
2. **Model** – A realistic layered conductivity model (air / seawater / marine sediments / bedrock) plus a compact highly-conductive target representing a metallic shipwreck.
3. **Survey** – A horizontal electric dipole source towed 5 m above the seafloor with an inline receiver array; four frequencies (0.1–5 Hz).
4. **Simulation** – Finite-volume FDEM simulation with both the full model and a target-free background.
5. **Analysis** – Amplitude, phase, normalised anomaly, and multi-frequency anomaly maps all confirm that the EM method can detect the buried archaeological target.

### Next steps
- Swap the toy analytical model for a real bathymetric/geological dataset.
- Perform **sensitivity analysis** to determine the minimum detectable target size.
- Run **gradient-based inversion** (SimPEG `InvProblem`) to recover the conductivity model from synthetic data.
- Export results to **VTK** for 3-D visualisation in ParaView.